# 00 Mount Mineral Photos Dataset

**AI-Assisted Mineral Identification — Capstone, Fall 2026**

Downloads the Mineral Photos dataset from Kaggle and copies them to a shared Google Drive folder.

What it does:
1. Sets the Kaggle credentials for kagglehub.
2. Uploads the dataset to the shared Drive folder.
3. Groups and verifies image categories (to compare both folders data and mineral-trier).

In [ ]:
import os
import shutil
from google.colab import drive
import kagglehub

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Set credentials
os.environ['KAGGLE_USERNAME'] = "<Username>"
os.environ['KAGGLE_KEY'] = "<Key>"

# 3. Download dataset using kagglehub
download_path = kagglehub.dataset_download("floriangeillon/mineral-photos")

# 4. Copy dataset to Google Drive folder
SHARED_DRIVE_PATH = "/content/drive/MyDrive/CSE_Capstone_Data/mineral-photos"
os.makedirs(SHARED_DRIVE_PATH, exist_ok=True)
shutil.copytree(download_path, SHARED_DRIVE_PATH, dirs_exist_ok=True)
print("Transfer complete!")

Mounted at /content/drive


100%|██████████| 14.8G/14.8G [02:23<00:00, 111MB/s]

Extracting files...


Transfer complete!


In [ ]:
!ls "/content/drive/MyDrive/CSE_Capstone_Data/mineral-photos"

data  mineral-trier


In [ ]:
import os
import pandas as pd
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Dataset directory
DATASET_PATH = "/content/drive/MyDrive/CSE_Capstone_Data/mineral-photos"

IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tif', '.tiff')
records = []

# Walk through all directories and subdirectories
for root, _, files in os.walk(DATASET_PATH):
    valid_images = [f for f in files if f.lower().endswith(IMAGE_EXTENSIONS)]

    if valid_images:
        rel_path = os.path.relpath(root, DATASET_PATH)
        path_parts = rel_path.split(os.sep)

        # Identify parent directory (e.g., 'data' or 'mineral-trier') and class name
        section = path_parts[0] if len(path_parts) > 1 else "Root"
        category = path_parts[-1]

        # Calculate total folder size
        folder_bytes = sum(os.path.getsize(os.path.join(root, f)) for f in valid_images)
        folder_mb = round(folder_bytes / (1024 * 1024), 2)

        records.append({
            "Section": section,
            "Mineral Class": category,
            "Image Count": len(valid_images),
            "Size (MB)": folder_mb,
            "Relative Path": rel_path
        })

df = pd.DataFrame(records)

if not df.empty:
    # Sort and re-index
    df = df.sort_values(by=["Section", "Mineral Class"]).reset_index(drop=True)

    # Full Element Count Table
    print("### DETAILED ELEMENT COUNT TABLE\n")
    print(df[["Section", "Mineral Class", "Image Count", "Size (MB)"]].to_markdown(index=False))

    # Section Aggregates
    print("\n### SECTION SUMMARY\n")
    summary = df.groupby("Section").agg(
        Total_Classes=("Mineral Class", "count"),
        Total_Images=("Image Count", "sum"),
        Total_MB=("Size (MB)", "sum")
    ).reset_index()
    summary["Total_GB"] = (summary["Total_MB"] / 1024).round(2)
    print(summary.to_markdown(index=False))

else:
    print(f"No image files found in '{DATASET_PATH}'. Verify the folder path.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
### DETAILED ELEMENT COUNT TABLE

| Section       | Mineral Class   |   Image Count |   Size (MB) |
|:--------------|:----------------|--------------:|------------:|
| data          | azurite         |          1250 |      419.41 |
| data          | baryte          |          2211 |      683.9  |
| data          | beryl           |          1615 |      418.13 |
| data          | calcite         |          6574 |     1617.89 |
| data          | cerussite       |          1367 |      488.04 |
| data          | copper          |          1429 |      445.88 |
| data          | fluorite        |          5669 |     1560.44 |
| data          | gypsum          |          1351 |      347.59 |
| data          | hematite        |          1284 |      456.81 |
| data          | malachite       |          1607 |      574.13 |
| data          | pyrite          |          